# Run APPS ground-truth solutions on Modal

This demo samples **3 filtered questions**, evaluates **1 qualifying reference answer per question**, and runs **all supplied cases** with the [official APPS evaluator](https://github.com/hendrycks/apps/blob/b45c0ed78517a3a6492eb77b21cffbb79b1096f1/eval/testing_util.py). Source executes only in fresh [Modal Sandboxes](https://modal.com/docs/guide/sandboxes).

Use the `stego` kernel and install `requirements-cpu.txt` (or the GPU requirements). The kernel must inherit `MODAL_TOKEN_ID` and `MODAL_TOKEN_SECRET`, or an existing Modal login, and `STEGO_ARTIFACTS_DIR`. Credentials stay local; no secrets are passed to sandboxes. The first run builds a Python 3.10 image with the evaluator's legacy dependency `pyext==0.6` and NumPy. This uses billable Modal CPU resources.

The sample size and timeouts below are editable. Case timeouts use APPS's signal alarm; the total solution deadline can stop a large test suite early. Ground truth is supplied by APPS, so failures are reported rather than assumed impossible.

In [1]:
import asyncio
import json
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

from pydantic import BaseModel, ConfigDict, Field

repo_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "ciphers/variable_naming_in_python_v2/data/modal_apps.py").is_file())
os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

## Check Modal before loading APPS

The next cell finds the configured evaluator App and logs whether it is reused or created. It then runs `print("Hello world:", 2 + 2)` in a temporary Python sandbox and checks for `Hello world: 4`. The App remains available for the evaluations below; the sandbox is terminated and detached even if execution fails.

This checks authentication, sandbox creation, and remote Python execution using your normal Modal credentials. It does not download APPS or install/run its evaluator. The first smoke test may build its minimal Python image. If this cell fails, resolve that error before continuing. Restart the kernel after changing credentials because the SDK caches its client.


In [ ]:
import modal
from modal.exception import NotFoundError

from ciphers.variable_naming_in_python_v2.data.modal_apps import EVALUATOR_REVISION, ModalAppsConfig, evaluate_on_modal

modal_config = ModalAppsConfig(case_timeout_s=4, solution_timeout_s=120, memory_mb=1024)

print(f"Looking up Modal App {modal_config.app_name!r}...", flush=True)
try:
    modal_app = await modal.App.lookup.aio(modal_config.app_name)
except NotFoundError:
    modal_app = await modal.App.lookup.aio(modal_config.app_name, create_if_missing=True)
    print(f"Created Modal App {modal_config.app_name!r} ({modal_app.app_id})", flush=True)
else:
    print(f"Reusing Modal App {modal_config.app_name!r} ({modal_app.app_id})", flush=True)

print("Starting hello-world sandbox...", flush=True)
with modal.enable_output():
    smoke_sandbox = await modal.Sandbox.create.aio(
        app=modal_app,
        image=modal.Image.debian_slim(python_version="3.10"),
        timeout=60,
        cpu=1,
        memory=256,
        block_network=True,
    )
try:
    smoke_process = await smoke_sandbox.exec.aio(
        "python", "-c", "print('Hello world:', 2 + 2)", timeout=10,
    )
    smoke_exit_code = await smoke_process.wait.aio()
    smoke_stdout = await smoke_process.stdout.read.aio()
    smoke_stderr = await smoke_process.stderr.read.aio()
    if smoke_exit_code != 0 or smoke_stdout.strip() != "Hello world: 4":
        raise RuntimeError(
            f"Modal smoke test failed (exit={smoke_exit_code}): "
            f"stdout={smoke_stdout!r}, stderr={smoke_stderr!r}"
        )
    print(smoke_stdout, end="")
    print("Modal smoke test passed (this is not a value check).")
finally:
    try:
        await smoke_sandbox.terminate.aio()
    finally:
        await smoke_sandbox.detach.aio()


/opt/miniconda3/envs/stego/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Looking up Modal App 'stego-apps-evaluation'...
Reusing Modal App 'stego-apps-evaluation' (ap-Dn6bSxewkirvWSLpl2SICY)
Starting hello-world sandbox...
Hello world: 4
Modal smoke test passed.


In [3]:
code_for_probe="""
def sum_array(arr):
    if len(arr or []) <= 2:
        return 0

    odd = len(arr) % 2

    if odd:
        total = lowest = highest = arr[0]
    else:
        total, lowest, highest = 0, float('inf'), -float('inf')

    for a, b in zip(arr[odd::2], arr[1 + odd::2]):
        total += a + b
        if a > b:
            lowest = min(b, lowest)
            highest = max(a, highest)
        else:
            lowest = min(a, lowest)
            highest = max(b, highest)

    return total - lowest - highest
""".strip()

In [ ]:
probe_input = [6, 2, 1, 8, 10]
print(f"Running code_for_probe on Modal with input {probe_input!r}...", flush=True)
with modal.enable_output():
    probe_sandbox = await modal.Sandbox.create.aio(
        app=modal_app,
        image=modal.Image.debian_slim(python_version="3.10"),
        timeout=60,
        cpu=1,
        memory=256,
        block_network=True,
    )
try:
    probe_process = await probe_sandbox.exec.aio(
        "python",
        "-c",
        code_for_probe + f"\nprint(repr(sum_array({probe_input!r})))\n",
        timeout=10,
    )
    probe_exit_code = await probe_process.wait.aio()
    probe_stdout = await probe_process.stdout.read.aio()
    probe_stderr = await probe_process.stderr.read.aio()
    if probe_exit_code != 0:
        raise RuntimeError(
            f"code_for_probe failed (exit={probe_exit_code}): "
            f"stdout={probe_stdout!r}, stderr={probe_stderr!r}"
        )
    print("Expect answer 16 in stdout")
    print("stdout:", probe_stdout, end="" if probe_stdout.endswith("\n") else "\n")
    if probe_stderr:
        print("stderr:", probe_stderr)
    print("code_for_probe Modal run finished.")
finally:
    try:
        await probe_sandbox.terminate.aio()
    finally:
        await probe_sandbox.detach.aio()


Running code_for_probe on Modal with input [6, 2, 1, 8, 10]...
stdout: 16
code_for_probe Modal run finished.


In [12]:
from importlib import reload

from ciphers.variable_naming_in_python_v2.data import modal_apps as modal_apps_module
from ciphers.variable_naming_in_python_v2.data.apps import AppsTestCases

reload(modal_apps_module)
evaluate_on_modal = modal_apps_module.evaluate_on_modal

probe_cases = AppsTestCases(
    inputs=[[probe_input]],
    outputs=[[16]],
    fn_name="sum_array",
)
probe_modal_config = modal_config.model_copy(update={"max_log_chars": 100_000})
print(f"Running code_for_probe through evaluate_on_modal with 1 case", flush=True)
print("--- Exact probe source submitted to Modal ---", flush=True)
print(code_for_probe, end="", flush=True)
print("\n--- End probe source ---", flush=True)
# Keep the synchronous grader off Jupyter's event loop.
verdict = await asyncio.to_thread(evaluate_on_modal, code_for_probe, probe_cases, probe_modal_config)
print(f"  {verdict.status}: {verdict.passed_tests}/{verdict.num_tests} cases passed")
print("  Raw APPS verdicts:", verdict.raw_results)
print("--- verdict.error ---")
print(verdict.error)
print("--- verdict.logs ---")
print(verdict.logs)


Running code_for_probe through evaluate_on_modal with 1 case
--- Exact probe source submitted to Modal ---
def sum_array(arr):
    if len(arr or []) <= 2:
        return 0

    odd = len(arr) % 2

    if odd:
        total = lowest = highest = arr[0]
    else:
        total, lowest, highest = 0, float('inf'), -float('inf')

    for a, b in zip(arr[odd::2], arr[1 + odd::2]):
        total += a + b
        if a > b:
            lowest = min(b, lowest)
            highest = max(a, highest)
        else:
            lowest = min(a, lowest)
            highest = max(b, highest)

    return total - lowest - highest
--- End probe source ---
  runner_error: 0/1 cases passed
UnsupportedOperation: fileno

  Raw APPS verdicts: []


In [7]:
from ciphers.variable_naming_in_python_v2.data.apps import AppsConfig, AppsTestCases, load_apps


class DemoConfig(BaseModel):
    model_config = ConfigDict(extra="forbid")
    num_problems: int = Field(default=3, ge=1)
    solutions_per_problem: int = Field(default=1, ge=1)
    seed: int = 42


demo = DemoConfig()
apps_config = AppsConfig()  # introductory/train, min_lines=20, min_tests=10
problems = load_apps(apps_config)
sample = problems.shuffle(seed=demo.seed).select(range(min(demo.num_problems, len(problems))))
if not len(sample):
    raise ValueError("No APPS problems match the loader filters")
print(f"Selected {len(sample)} of {len(problems)} eligible questions")

Skipped 1 malformed APPS rows


Selected 3 of 223 eligible questions


Run the following cell to submit solutions. Each row's `input_output` contains paired `inputs`/`outputs` plus `fn_name`: null means standard input/output; a string names a function or `Solution` method. The helper consumes that schema without changing the supplied cases.

Verdicts preserve APPS's conventions: `true` passes, `false` is a wrong answer, `-1` combines runtime error and per-case timeout, and `-2` denotes compilation/initialization failure. APPS applies permissive numeric and unordered comparisons; this is an evaluator-compatibility demo, not a stricter judge. Modal process deadlines are reported separately as `timeout`. Authentication/build/transport errors raise instead of being counted as wrong answers.

In [8]:
records = []
for problem in sample:
    cases = AppsTestCases.from_dataset_value(problem["input_output"])
    for solution_index, source in enumerate(problem["solutions"][: demo.solutions_per_problem]):
        print(f"Running APPS {apps_config.split}/{problem['problem_id']}, reference {solution_index}, {problem['num_tests']} cases", flush=True)
        print("--- Exact reference source submitted to Modal ---", flush=True)
        print(source, end="", flush=True)
        print("\n--- End reference source ---", flush=True)
        # Keep the synchronous grader off Jupyter's event loop.
        verdict = await asyncio.to_thread(evaluate_on_modal, source, cases, modal_config)
        records.append(
            {
                "problem_id": problem["problem_id"],
                "url": problem["url"],
                "solution_index": solution_index,
                "source": source,
                "input_output": problem["input_output"],
                "verdict": verdict.model_dump(),
            }
        )
        print(f"  {verdict.status}: {verdict.passed_tests}/{verdict.num_tests} cases passed")
        if verdict.error or verdict.logs:
            print(verdict.error or "")
            print(verdict.logs)
        print("  Raw APPS verdicts:", verdict.raw_results)

Running APPS train/3094, reference 0, 10 cases
--- Exact reference source submitted to Modal ---
def sum_array(arr):
    if len(arr or []) <= 2:
        return 0

    odd = len(arr) % 2

    if odd:
        total = lowest = highest = arr[0]
    else:
        total, lowest, highest = 0, float('inf'), -float('inf')

    for a, b in zip(arr[odd::2], arr[1 + odd::2]):
        total += a + b
        if a > b:
            lowest = min(b, lowest)
            highest = max(a, highest)
        else:
            lowest = min(a, lowest)
            highest = max(b, highest)

    return total - lowest - highest

--- End reference source ---
  runner_error: 0/10 cases passed
UnsupportedOperation: fileno

  Raw APPS verdicts: []
Running APPS train/3520, reference 0, 21 cases
--- Exact reference source submitted to Modal ---
def step(g, m, n):
    return next(([a, a+g] for a in range(m, n-g+1) if is_prime(a) and is_prime(a+g)), None)


def is_prime(n):
    factors = 0
    for k in (2, 3):
        whi

In [9]:
print(f"Fully passing solutions: {sum(record['verdict']['status'] == 'passed' for record in records)}/{len(records)}")
artifact_root = (repo_root / os.environ["STEGO_ARTIFACTS_DIR"]).resolve()
output_dir = artifact_root / "datasets/apps/modal-ground-truth"
output_dir.mkdir(parents=True, exist_ok=True)
report_path = output_dir / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + ".json")
report = {
    "demo": demo.model_dump(),
    "apps_config": apps_config.model_dump(mode="json"),
    "modal_config": modal_config.model_dump(),
    "evaluator_revision": EVALUATOR_REVISION,
    "records": records,
}
report_path.write_text(json.dumps(report, indent=2))
print("Saved report:", report_path)

Fully passing solutions: 0/3
Saved report: /Users/4gate/git/StegoICMLMechInterp2026/artifacts/datasets/apps/modal-ground-truth/20260915T024531939668Z.json
